# Train Event Classifier

This notebook trains a first temporal classifier from clips exported by `tools/export_events/export_event_clips.py`.

Expected input:

```txt
data/exports/events/
  class_names.json
  metadata.csv
  clips/train/<event>/*.mp4
  clips/val/<event>/*.mp4
```

## 1. Runtime Check

In Colab, choose `Runtime > Change runtime type > GPU` before training.

In [ ]:
import sys
import platform

print("python", sys.version)
print("platform", platform.platform())

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")

## 2. Install Dependencies

In [ ]:
%pip install -q opencv-python pandas tqdm

## 3. Provide Dataset

Upload a zipped event export named `events.zip`, or mount Drive and set `DATA_ROOT` manually.

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("events.zip")
extract_dir = Path("/content/events")
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extract_dir)

matches = sorted(extract_dir.rglob("metadata.csv"))
DATA_ROOT = matches[0].parent if matches else extract_dir
METADATA_CSV = DATA_ROOT / "metadata.csv"
CLASS_NAMES_JSON = DATA_ROOT / "class_names.json"

print("data root:", DATA_ROOT)
print("metadata exists:", METADATA_CSV.exists())
print("class names exists:", CLASS_NAMES_JSON.exists())

## 4. Dataset And Model

This baseline samples 16 frames per clip and fine-tunes a torchvision `r3d_18` video model.

In [ ]:
import json
import cv2
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models.video import r3d_18, R3D_18_Weights

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_FRAMES = 16
IMAGE_SIZE = 112
BATCH_SIZE = 4

class_names = json.loads(CLASS_NAMES_JSON.read_text())
class_names = [class_names[str(index)] for index in range(len(class_names))]
num_classes = len(class_names)
print(class_names)

metadata = pd.read_csv(METADATA_CSV)
print(metadata.groupby(["split", "label"]).size())

class EventClipDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        clip_path = DATA_ROOT / row.clip_path
        frames = load_clip(clip_path, NUM_FRAMES, IMAGE_SIZE)
        label = int(row.class_id)
        return frames, torch.tensor(label, dtype=torch.long)

def load_clip(path, num_frames, image_size):
    capture = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (image_size, image_size), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    capture.release()

    if not frames:
        frames = [torch.zeros(image_size, image_size, 3, dtype=torch.uint8).numpy()]

    if len(frames) < num_frames:
        frames.extend([frames[-1]] * (num_frames - len(frames)))

    if len(frames) > num_frames:
        indices = torch.linspace(0, len(frames) - 1, num_frames).round().long().tolist()
        frames = [frames[i] for i in indices]

    tensor = torch.tensor(frames, dtype=torch.float32) / 255.0
    tensor = tensor.permute(3, 0, 1, 2)  # C, T, H, W
    mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
    std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
    return (tensor - mean) / std

train_rows = metadata[metadata.split == "train"]
val_rows = metadata[metadata.split == "val"]
train_loader = DataLoader(EventClipDataset(train_rows), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(EventClipDataset(val_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

## 5. Train

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    for clips, labels in tqdm(loader, leave=False):
        clips = clips.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(training):
            logits = model(clips)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += float(loss) * labels.size(0)
        correct += int((logits.argmax(dim=1) == labels).sum())
        total += labels.size(0)

    if total == 0:
        return {"loss": None, "acc": None}
    return {"loss": total_loss / total, "acc": correct / total}

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    print(f"epoch {epoch}: train={train_metrics} val={val_metrics}")

torch.save({"model": model.state_dict(), "class_names": class_names}, "/content/event_classifier_r3d18.pt")